# Chapter 16 Companion Notebook: Explainability, Fairness, and Regulation

This notebook reproduces every worked example in Chapter 16 from scratch, in the order they appear in the chapter text, including a calibration-in-the-large check by group (Section 6b) completing the fairness audit's three-way impossibility tension.

---

**© 2026 Wulin Suo. All rights reserved.** This notebook is a companion to the draft manuscript *AI in Finance* and is provided for personal, educational use. No part of this notebook may be reproduced, distributed, or transmitted in any form or by any means without the prior written permission of the author, except for brief quotations in a review. Contact: Wulin.Suo@Queensu.ca

## 1. Permutation feature importance (Chapter 13 fraud-scoring model)

In [1]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.array([3.1, 0.2, 2.8, 1.9, 0.4, 3.4, 0.1, 0.6, 1.2, 0.3])
d = np.array([8.0, 0.1, 6.5, 6.0, 0.2, 9.2, 0.4, 0.1, 0.3, 0.2])
n = np.array([1, 0, 1, 0, 0, 1, 0, 1, 1, 0])
actual = np.array([1, 0, 1, 0, 0, 1, 0, 0, 1, 0])
b0, b1, b2, b3 = -2.57, 0.83, 0.15, 0.82

def predict(zc, dc, nc):
    p = sigmoid(b0 + b1 * zc + b2 * dc + b3 * nc)
    return p, (p >= 0.5).astype(int)

def metrics(pred, actual):
    tp = np.sum((pred == 1) & (actual == 1))
    fp = np.sum((pred == 1) & (actual == 0))
    fn = np.sum((pred == 0) & (actual == 1))
    tn = np.sum((pred == 0) & (actual == 0))
    acc = (tp + tn) / len(actual)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else float('nan')
    return acc, prec, rec, f1

p_base, pred_base = predict(z, d, n)
base_metrics = metrics(pred_base, actual)
print("baseline predictions:", pred_base)
print("baseline acc/prec/rec/F1:", [round(v, 3) if isinstance(v, float) else v for v in base_metrics])

for name, col, args in [
    ("amount z-score", z, lambda c: predict(c, d, n)),
    ("distance", d, lambda c: predict(z, c, n)),
    ("late-night", n, lambda c: predict(z, d, c)),
]:
    col_perm = col[::-1]
    p_perm, pred_perm = args(col_perm)
    m = metrics(pred_perm, actual)
    print(f"permuted {name}: predictions={pred_perm} acc={m[0]:.3f} prec={m[1]:.3f} rec={m[2]:.3f} F1={m[3]}")

baseline predictions: [1 0 1 0 0 1 0 0 0 0]
baseline acc/prec/rec/F1: [np.float64(0.9), np.float64(1.0), np.float64(0.75), np.float64(0.857)]
permuted amount z-score: predictions=[0 0 0 0 1 0 0 1 0 1] acc=0.300 prec=0.000 rec=0.000 F1=nan
permuted distance: predictions=[1 0 1 0 0 1 0 0 0 0] acc=0.900 prec=1.000 rec=0.750 F1=0.8571428571428571
permuted late-night: predictions=[1 0 1 0 0 1 0 0 0 0] acc=0.900 prec=1.000 rec=0.750 F1=0.8571428571428571


## 2. Partial dependence of predicted fraud probability on amount z-score

In [2]:
grid = np.array([-1, 0, 1, 2, 3, 4])
pd_vals = np.array([sigmoid(b0 + b1 * g + b2 * d + b3 * n).mean() for g in grid])
print("z grid:", grid)
print("partial dependence:", pd_vals.round(4))

for i in range(len(grid) - 1):
    if (pd_vals[i] - 0.5) * (pd_vals[i+1] - 0.5) < 0:
        frac = (0.5 - pd_vals[i]) / (pd_vals[i+1] - pd_vals[i])
        cross = grid[i] + frac * (grid[i+1] - grid[i])
        print(f"crosses 0.5 near z = {cross:.3f}")

z grid: [-1  0  1  2  3  4]
partial dependence: [0.0954 0.1845 0.3195 0.4888 0.663  0.8067]
crosses 0.5 near z = 2.065


## 3. Exact Shapley (additive) decomposition, credit-scoring model (Chapter 9)

In [3]:
dti_pop = np.array([35, 20, 40, 15, 30, 18])
ch_pop = np.array([2, 8, 1, 10, 3, 7])
dti_bar, ch_bar = dti_pop.mean(), ch_pop.mean()
print("reference population mean DTI:", dti_bar, " mean credit history:", ch_bar)

beta0, beta1, beta2 = -12.15176948, 0.52992476, -0.23135204  # exact sklearn fit, Chapter 9
dti_x, ch_x = 28, 4
z_x = beta0 + beta1 * dti_x + beta2 * ch_x
z_baseline = beta0 + beta1 * dti_bar + beta2 * ch_bar
contrib_dti = beta1 * (dti_x - dti_bar)
contrib_ch = beta2 * (ch_x - ch_bar)

print("z (applicant):", round(z_x, 4), " z (baseline):", round(z_baseline, 4))
print("contribution of DTI:", round(contrib_dti, 4))
print("contribution of credit history:", round(contrib_ch, 4))
print("check: baseline + contributions =", round(z_baseline + contrib_dti + contrib_ch, 4), " vs z_x =", round(z_x, 4))

p_x, p_baseline = sigmoid(z_x), sigmoid(z_baseline)
print("Pr(default) applicant:", round(p_x, 4), " baseline:", round(p_baseline, 4))
print("score applicant:", round(850 - 550 * p_x, 1), " baseline:", round(850 - 550 * p_baseline, 1))

reference population mean DTI: 26.333333333333332  mean credit history: 5.166666666666667
z (applicant): 1.7607  z (baseline): 0.6076
contribution of DTI: 0.8832
contribution of credit history: 0.2699
check: baseline + contributions = 1.7607  vs z_x = 1.7607
Pr(default) applicant: 0.8533  baseline: 0.6474
score applicant: 380.7  baseline: 493.9


## 4. Counterfactual explanation: minimum change to reach loan approval

In [4]:
approval_score_threshold = 620
p_threshold = (850 - approval_score_threshold) / 550
z_threshold = np.log(p_threshold / (1 - p_threshold))
print("approval requires Pr(default) <=", round(p_threshold, 4), " i.e. z <=", round(z_threshold, 4))

dti_new = (z_threshold - beta0 - beta2 * ch_x) / beta1
print("counterfactual DTI (holding credit history = 4 fixed):", round(dti_new, 3),
      " change:", round(dti_new - dti_x, 3))

approval requires Pr(default) <= 0.4182  i.e. z <= -0.3302
counterfactual DTI (holding credit history = 4 fixed): 24.054  change: -3.946


## 5. LIME local surrogate on Chapter 6's 2-ReLU feedforward network

In [5]:
W1 = np.array([[0.4, -0.2], [0.1, 0.3]])
b1_nn = np.array([0.05, -0.1])
W2 = np.array([0.6, -0.5])
b2_nn = 0.1

def net_forward(x):
    h = np.maximum(0, W1 @ x + b1_nn)
    z2 = W2 @ h + b2_nn
    return sigmoid(z2), z2, h

x0 = np.array([1.0, 0.5])
y0, z0, h0 = net_forward(x0)
print("x0:", x0, " h0:", h0, " z0:", round(z0, 4), " y0:", round(y0, 4))

dz_dx = W2 @ W1
dy_dx = y0 * (1 - y0) * dz_dx
print("analytical local gradient:", dy_dx.round(4))

perturbations = np.array([
    [1.1, 0.5], [0.9, 0.5], [1.0, 0.6], [1.0, 0.4],
    [1.1, 0.6], [0.9, 0.4], [1.1, 0.4], [0.9, 0.6],
])
ys = np.array([net_forward(xp)[0] for xp in perturbations])
X_lime = np.column_stack([np.ones(len(perturbations)), perturbations])
coef, _, _, _ = np.linalg.lstsq(X_lime, ys, rcond=None)
print("LIME surrogate (intercept, slope1, slope2):", coef.round(4))

y_surrogate_x0 = coef @ np.array([1.0, x0[0], x0[1]])
print("surrogate prediction at x0:", round(y_surrogate_x0, 4), " true y0:", round(y0, 4))

x_far = np.array([1.0, -0.1])
y_far, z_far, h_far = net_forward(x_far)
y_surrogate_far = coef @ np.array([1.0, x_far[0], x_far[1]])
print("far point x=(1.0,-0.1): true y =", round(y_far, 4), " surrogate predicts:", round(y_surrogate_far, 4))

x0: [1.  0.5]  h0: [0.35 0.15]  z0: 0.235  y0: 0.5585
analytical local gradient: [ 0.0469 -0.0666]
LIME surrogate (intercept, slope1, slope2): [ 0.5449  0.0468 -0.0666]
surrogate prediction at x0: 0.5585  true y0: 0.5585
far point x=(1.0,-0.1): true y = 0.5944  surrogate predicts: 0.5984


## 6. Fairness audit on a synthetic two-group applicant pool

In [6]:
alpha_dti = np.array([15, 22, 30, 18]); alpha_ch = np.array([9, 6, 3, 8]); alpha_default = np.array([0, 1, 1, 0])
beta_dti = np.array([25, 33, 20, 38]); beta_ch = np.array([4, 2, 5, 1]); beta_default = np.array([0, 1, 0, 1])

def audit(dti, ch, will_default, label):
    zg = beta0 + beta1 * dti + beta2 * ch
    pg = sigmoid(zg)
    approve = pg <= p_threshold
    approval_rate = approve.mean()
    mask = will_default == 1
    fnr = approve[mask].mean()
    print(f"{label}: Pr(default)={pg.round(3)} approve={approve} approval_rate={approval_rate:.3f} FNR={fnr:.3f}")
    return approval_rate, fnr

ar_a, fnr_a = audit(alpha_dti, alpha_ch, alpha_default, "Group Alpha")
ar_b, fnr_b = audit(beta_dti, beta_ch, beta_default, "Group Beta")
air = min(ar_a, ar_b) / max(ar_a, ar_b)
print("adverse impact ratio:", round(air, 3))

Group Alpha: Pr(default)=[0.002 0.132 0.955 0.011] approve=[ True  True False  True] approval_rate=0.750 FNR=0.500
Group Beta: Pr(default)=[0.543 0.992 0.062 1.   ] approve=[False False  True False] approval_rate=0.250 FNR=0.000
adverse impact ratio: 0.333


### 6b. Calibration-in-the-large by group (Section 16.6)

Both groups share the same actual default rate; check whether the model's average predicted probability matches it within each group separately.

In [7]:
p_alpha = sigmoid(beta0 + beta1 * alpha_dti + beta2 * alpha_ch)
p_beta = sigmoid(beta0 + beta1 * beta_dti + beta2 * beta_ch)

alpha_actual_rate = alpha_default.mean()
beta_actual_rate = beta_default.mean()

print("Group Alpha predicted probabilities:", p_alpha.round(4))
print(f"Group Alpha: avg predicted = {p_alpha.mean():.4f}, actual default rate = {alpha_actual_rate:.4f}")
print()
print("Group Beta predicted probabilities:", p_beta.round(4))
print(f"Group Beta: avg predicted = {p_beta.mean():.4f}, actual default rate = {beta_actual_rate:.4f}")

Group Alpha predicted probabilities: [0.0019 0.1322 0.9549 0.0114]
Group Alpha: avg predicted = 0.2751, actual default rate = 0.5000

Group Beta predicted probabilities: [0.5426 0.9924 0.0624 0.9996]
Group Beta: avg predicted = 0.6492, actual default rate = 0.5000


## 7. Adversarial 'structuring' perturbation on the fraud-scoring model

In [8]:
d1, n1 = 8.0, 1
p1, _ = predict(np.array([3.1]), np.array([d1]), np.array([n1]))
print("transaction 1 original amount z-score = 3.1, predicted prob =", round(p1[0], 4))

z_flip = (0 - b0 - b2 * d1 - b3 * n1) / b1
print("amount z-score needed to flip the decision (holding distance, late-night fixed):", round(z_flip, 4))
print("required reduction:", round(3.1 - z_flip, 4))

transaction 1 original amount z-score = 3.1, predicted prob = 0.8832
amount z-score needed to flip the decision (holding distance, late-night fixed): 0.6627
required reduction: 2.4373


## 8. Population stability index for score-distribution drift

In [9]:
baseline_dist = np.array([0.50, 0.25, 0.15, 0.10])
current_dist = np.array([0.34, 0.29, 0.22, 0.15])
psi_terms = (current_dist - baseline_dist) * np.log(current_dist / baseline_dist)
print("bin contributions:", psi_terms.round(4))
print("total PSI:", round(psi_terms.sum(), 4))

bin contributions: [0.0617 0.0059 0.0268 0.0203]
total PSI: 0.1147


## 9. Algorithmic recourse cost: debt-to-income vs. credit-history paths

In [10]:
ch_new = (z_threshold - beta0 - beta1 * dti_x) / beta2
delta_dti = dti_new - dti_x
delta_ch = ch_new - ch_x
w_dti, w_ch = 1.0, 5.0
cost_dti = abs(delta_dti) * w_dti
cost_ch = abs(delta_ch) * w_ch

print("counterfactual credit history (holding DTI=28 fixed):", round(ch_new, 3), " change:", round(delta_ch, 3))
print("debt-to-income path: delta =", round(delta_dti, 3), " cost =", round(cost_dti, 3))
print("credit-history path: delta =", round(delta_ch, 3), " cost =", round(cost_ch, 3))

counterfactual credit history (holding DTI=28 fixed): 13.038  change: 9.038
debt-to-income path: delta = -3.946  cost = 3.946
credit-history path: delta = 9.038  cost = 45.19


## 10. Integrated gradients on Chapter 6's 2-ReLU network

In [11]:
x_prime = np.array([0.0, 0.0])
p_prime, z_prime, h_prime = net_forward(x_prime)
print("y(x'):", round(p_prime, 4), " z2(x'):", round(z_prime, 4), " h(x'):", h_prime.round(4))
print("y(x0):", round(y0, 4), " completeness target:", round(y0 - p_prime, 4))

t_break = 0.1 / 0.25
print("breakpoint t where hidden unit 2 activates:", t_break)

def seg_integral(t0, t1, dzdx1, dzdx2, b_slope, a_intercept):
    z_t0 = a_intercept + b_slope * t0
    z_t1 = a_intercept + b_slope * t1
    integral_sigmoid_prime = (sigmoid(z_t1) - sigmoid(z_t0)) / b_slope
    return dzdx1 * integral_sigmoid_prime, dzdx2 * integral_sigmoid_prime

a1 = 0.6 * 0.05 + b2_nn
b1_seg = 0.6 * (0.4 * 1.0 - 0.2 * 0.5)
ig1_seg1, ig2_seg1 = seg_integral(0, t_break, 0.24, -0.12, b1_seg, a1)

xt_break = x_prime + t_break * (x0 - x_prime)
z_at_break = W2 @ np.maximum(0, W1 @ xt_break + b1_nn) + b2_nn
b2_seg = 0.6 * (0.4 * 1.0 - 0.2 * 0.5) - 0.5 * (0.1 * 1.0 + 0.3 * 0.5)
ig1_seg2, ig2_seg2 = seg_integral(t_break, 1.0, 0.19, -0.27, b2_seg, z_at_break - b2_seg * t_break)

IG1 = (x0[0] - x_prime[0]) * (ig1_seg1 + ig1_seg2)
IG2 = (x0[1] - x_prime[1]) * (ig2_seg1 + ig2_seg2)
print("IG1:", round(IG1, 4), " IG2:", round(IG2, 4), " sum:", round(IG1 + IG2, 4))

naive_attr = dy_dx * (x0 - x_prime)
print("naive gradient x delta:", naive_attr.round(4), " sum:", round(naive_attr.sum(), 4))

y(x'): 0.5325  z2(x'): 0.13  h(x'): [0.05 0.  ]
y(x0): 0.5585  completeness target: 0.026
breakpoint t where hidden unit 2 activates: 0.4
IG1: 0.052  IG2: -0.026  sum: 0.026
naive gradient x delta: [ 0.0469 -0.0333]  sum: 0.0136
